# 05 — Function Calling Across Providers — Practical Examples

**Covers**: OpenAI vs Anthropic vs Gemini formats, LiteLLM unified API, provider fallback

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()
print('✅ Setup ready. API keys loaded from .env')

## 1. OpenAI Function Calling

In [ ]:
from openai import OpenAI
client_oai = OpenAI()

TOOLS_OAI = [{'type':'function','function':{'name':'get_weather','description':'Get weather for a city','parameters':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}}]

def mock_weather(city): return f'{city}: 25°C, Sunny'

r = client_oai.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user','content':'Weather in Tokyo?'}],
    tools=TOOLS_OAI
)
msg = r.choices[0].message
if msg.tool_calls:
    tc = msg.tool_calls[0]
    args = json.loads(tc.function.arguments)  # ← Must json.loads()
    result = mock_weather(**args)
    print(f'OpenAI → Tool: {tc.function.name}, Args: {args}, ID: {tc.id}')
    print(f'Result: {result}')

## 2. Anthropic (Claude) Tool Use

In [ ]:
try:
    import anthropic
    client_claude = anthropic.Anthropic()
    
    TOOLS_CLAUDE = [{
        'name': 'get_weather',
        'description': 'Get weather for a city',
        'input_schema': {  # ← 'input_schema' not 'parameters'
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'City name'}},
            'required': ['city']
        }
    }]
    
    r = client_claude.messages.create(
        model='claude-3-5-haiku-20241022',
        max_tokens=200,
        tools=TOOLS_CLAUDE,
        messages=[{'role':'user','content':'Weather in Tokyo?'}]
    )
    
    for block in r.content:
        if block.type == 'tool_use':
            print(f'Claude → Tool: {block.name}')
            print(f'Args: {block.input}')     # ← Already a dict! No json.loads needed
            print(f'ID: {block.id}')          # ← 'id' not 'tool_call_id'
            print('Note: result goes back as role=user with tool_result content block')
except Exception as e:
    print(f'Anthropic error (need ANTHROPIC_API_KEY): {e}')

## 3. LiteLLM — One API for All Providers

In [ ]:
try:
    import litellm
    
    TOOLS_LITELLM = [{
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get weather for a city',
            'parameters': {'type':'object','properties':{'city':{'type':'string'}},'required':['city']}
        }
    }]
    
    # Same code — different model strings
    for model_name in ['gpt-4o-mini']:
        try:
            r = litellm.completion(
                model=model_name,
                messages=[{'role':'user','content':'Weather in Paris?'}],
                tools=TOOLS_LITELLM
            )
            tc = r.choices[0].message.tool_calls
            print(f'{model_name}: tool_called={bool(tc)}')
            if tc:
                print(f'  → {tc[0].function.name}({json.loads(tc[0].function.arguments)})')
        except Exception as e:
            print(f'{model_name}: Error — {str(e)[:80]}')
except ImportError:
    print('pip install litellm')

## 4. Provider Fallback Chain

In [ ]:
def call_with_fallback(messages, tools):
    models = ['gpt-4o-mini', 'gpt-3.5-turbo']
    for model in models:
        try:
            from openai import OpenAI
            c = OpenAI()
            r = c.chat.completions.create(model=model, messages=messages, tools=tools, timeout=15)
            print(f'✅ Succeeded with: {model}')
            return r
        except Exception as e:
            print(f'❌ {model} failed: {str(e)[:60]}. Trying next...')
    raise RuntimeError('All models failed')

r = call_with_fallback(
    messages=[{'role':'user','content':'Weather in Berlin?'}],
    tools=TOOLS_OAI
)
print('Final answer available:', bool(r))